# CNN4 Q1 Final: MRI Classification with Feature Extraction

This notebook implements a comprehensive MRI classification pipeline combining state-of-the-art techniques:

1. **RFECV Feature Selection** - Recursive Feature Elimination with Cross-Validation
2. **Stacking Classifier** - Meta-learning ensemble approach
3. **K-Fold Cross-Validation** - For robust performance estimation
4. **Multiple Ensemble Strategies** - Combining diverse classifiers
5. **Calibrated Probabilities** - For reliable probability estimates

**Target Performance:** Val AUC > 0.72, Test AUC > 0.75

## 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import nibabel as nib

from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, precision_recall_fscore_support, confusion_matrix
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, 
    ExtraTreesClassifier, StackingClassifier, VotingClassifier
)
from sklearn.feature_selection import RFECV, SelectFromModel
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.base import TransformerMixin, BaseEstimator
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import json

np.random.seed(42)
torch.manual_seed(42)

## 2. Configuration and Paths

In [2]:
SCRIPT_DIR = Path('.').resolve()
SPLITS_DIR = SCRIPT_DIR / '../new_splits'
FEATURES_DIR = SCRIPT_DIR / '../../cnn4_features'
OUTPUT_DIR = SCRIPT_DIR / 'results_advanced'
OUTPUT_DIR.mkdir(exist_ok=True)

print("="*80)
print("CNN4 Q1 FINAL: With Feature Extraction")
print("="*80)

CNN4 Q1 FINAL: With Feature Extraction


## 3. CNN4 Model Architecture

Implementation of the ResidualNet3D backbone for deep feature extraction from 3D MRI scans.

### 3.1 Residual Module 3D

In [3]:
class ResidualModule3D(nn.Module):
    def __init__(self, inplanes, planes, stride=1):
        super().__init__()
        planes_4 = planes // 4
        self.bn1 = nn.BatchNorm3d(inplanes)
        self.relu1 = nn.LeakyReLU()
        self.conv1 = nn.Conv3d(inplanes, planes_4, 1, bias=False)
        self.bn2 = nn.BatchNorm3d(planes_4)
        self.relu2 = nn.LeakyReLU()
        self.conv2 = nn.Conv3d(planes_4, planes_4, 3, stride, 1, bias=False)
        self.bn3 = nn.BatchNorm3d(planes_4)
        self.relu3 = nn.LeakyReLU()
        self.conv3 = nn.Conv3d(planes_4, planes, 1, bias=False)
        self.conv4 = nn.Conv3d(inplanes, planes, 1, stride, bias=False)
        self.downsample = (inplanes != planes) or (stride != 1)
    
    def forward(self, x):
        residual = x
        out = self.relu1(self.bn1(x))
        out1 = out
        out = self.conv3(self.relu3(self.bn3(self.conv2(self.relu2(self.bn2(self.conv1(out)))))))
        if self.downsample:
            residual = self.conv4(out1)
        return out + residual

### 3.2 Residual Network 3D

In [4]:
class ResidualNet3D(nn.Module):
    def __init__(self, width_f=3):
        super().__init__()
        nf = [int(width_f * f) for f in [8, 16, 32, 64, 128]]
        self.features = nn.Sequential(
            nn.Conv3d(1, nf[0], 7, 2, 3, bias=False),
            nn.BatchNorm3d(nf[0]),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(3, 2, 1),
            ResidualModule3D(nf[0], nf[1]),
            ResidualModule3D(nf[1], nf[2], 2),
            ResidualModule3D(nf[2], nf[3], 2),
            ResidualModule3D(nf[3], nf[4], 2),
            ResidualModule3D(nf[4], nf[4]),
            ResidualModule3D(nf[4], nf[4]),
            nn.BatchNorm3d(nf[4]),
            nn.ReLU(inplace=True),
            nn.AvgPool3d(3, 1)
        )
    
    def forward(self, x):
        return self.features(x).view(x.size(0), -1)

### 3.3 CNN4 Classifier

In [5]:
class CNN4Classifier(nn.Module):
    def __init__(self, pretrained_path=None):
        super().__init__()
        self.backbone = ResidualNet3D(width_f=3)
        
        with torch.no_grad():
            self.feat_dim = self.backbone(torch.zeros(1, 1, 91, 109, 91)).shape[1]
        
        if pretrained_path and Path(pretrained_path).exists():
            ckpt = torch.load(pretrained_path, map_location='cpu', weights_only=False)
            sd = ckpt.get('state_dict', ckpt)
            sd = {k.replace('module.', ''): v for k, v in sd.items()}
            self.backbone.load_state_dict(sd, strict=False)
            print(f"  ✓ Loaded CNN4 pretrained weights ({self.feat_dim}D)")
        
        self.head = nn.Sequential(
            nn.Dropout(0.6),
            nn.Linear(self.feat_dim, 1)
        )
        
    def forward(self, x):
        feats = self.backbone(x)
        return self.head(feats).squeeze(-1)

## 4. Feature Extraction Function

In [6]:
def extract_features_for_split(model, split_name, device):
    print(f"  Extracting {split_name}...")
    
    df = pd.read_csv(SPLITS_DIR / f'{split_name}.csv')
    base = Path('../../outputs/standalone')
    
    features, paths_list = [], []
    
    model.eval()
    with torch.no_grad():
        for idx, row in df.iterrows():
            pid = row['participant_id']
            p = base / pid / 'ses-01' / f"{pid}_ses-01_preprocessed.nii.gz"
            
            if not p.exists():
                continue
            
            try:
                data = nib.load(str(p)).get_fdata().astype(np.float32)
                if data.max() > data.min():
                    data = (data - data.min()) / (data.max() - data.min())
                x = torch.from_numpy(data).unsqueeze(0).unsqueeze(0).float().to(device)
                
                feat = model.backbone(x).cpu().numpy()[0]
                
                features.append(feat)
                paths_list.append(str(p))
                
                if (idx + 1) % 50 == 0:
                    print(f"    {idx + 1}/{len(df)}...")
                
            except Exception as e:
                print(f"    Error {pid}: {e}")
    
    features = np.array(features)
    paths_list = np.array(paths_list)
    
    print(f"    Extracted: {features.shape}")
    
    output_path = FEATURES_DIR / f'mri_features_{split_name}_new.npz'
    np.savez(output_path, features=features, paths=paths_list)
    print(f"    Saved: {output_path}")
    
    return len(features)

## 5. Extract CNN4 Features

In [7]:
print("\n[0/6] Extracting CNN4 features from MRI scans...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device: {device}")

pretrained_path = '../../pretrained_weights/CNN4-weights-v1.0/ResNet3D_3x_0_cv_3.pth'
cnn4_model = CNN4Classifier(pretrained_path).to(device)

for split in ['train', 'val', 'test']:
    extract_features_for_split(cnn4_model, split, device)

print("  ✓ Feature extraction complete!")

del cnn4_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


[0/6] Extracting CNN4 features from MRI scans...
  Device: cuda
  ✓ Loaded CNN4 pretrained weights (768D)
  Extracting train...
    50/180...
    100/180...
    150/180...
    Extracted: (180, 768)
    Saved: D:\qtab dataset\anxiety_multimodal\new_mri\..\..\cnn4_features\mri_features_train_new.npz
  Extracting val...
    50/62...
    Extracted: (62, 768)
    Saved: D:\qtab dataset\anxiety_multimodal\new_mri\..\..\cnn4_features\mri_features_val_new.npz
  Extracting test...
    50/62...
    Extracted: (62, 768)
    Saved: D:\qtab dataset\anxiety_multimodal\new_mri\..\..\cnn4_features\mri_features_test_new.npz
  ✓ Feature extraction complete!


## 6. Load and Align Features

In [8]:
def load_features(split_df, npz_path):
    data = np.load(npz_path, allow_pickle=True)
    paths, features = data['paths'], data['features']
    
    def get_pid(p):
        for part in Path(str(p)).parts:
            if part.startswith('sub-'):
                return part
        return None
    
    path_to_idx = {get_pid(p): i for i, p in enumerate(paths)}
    
    aligned, labels, pids = [], [], []
    for pid in split_df['participant_id'].values:
        if pid in path_to_idx:
            aligned.append(features[path_to_idx[pid]])
            labels.append(split_df[split_df['participant_id']==pid]['internalizing_incident'].values[0])
            pids.append(pid)
    
    return np.array(aligned), np.array(labels), pids

In [9]:
print("\n[1/6] Loading data...")
train_df = pd.read_csv(SPLITS_DIR / 'train.csv')
val_df = pd.read_csv(SPLITS_DIR / 'val.csv')
test_df = pd.read_csv(SPLITS_DIR / 'test.csv')

X_train, y_train, pids_train = load_features(train_df, FEATURES_DIR / 'mri_features_train_new.npz')
X_val, y_val, pids_val = load_features(val_df, FEATURES_DIR / 'mri_features_val_new.npz')
X_test, y_test, pids_test = load_features(test_df, FEATURES_DIR / 'mri_features_test_new.npz')

print(f"  Train: {X_train.shape}, pos={y_train.sum()}")
print(f"  Val:   {X_val.shape}, pos={y_val.sum()}")
print(f"  Test:  {X_test.shape}, pos={y_test.sum()}")


[1/6] Loading data...
  Train: (180, 768), pos=24
  Val:   (62, 768), pos=7
  Test:  (62, 768), pos=7


## 7. Data Augmentation

In [10]:
print("\n[2/6] Data Augmentation...")

def augment_features(X, y, factor=3):
    idx_pos = np.where(y == 1)[0]
    X_pos, y_pos = X[idx_pos], y[idx_pos]
    
    X_aug, y_aug = [X], [y]
    for _ in range(factor - 1):
        noise = np.random.normal(0, 0.02, X_pos.shape)
        X_aug.append(X_pos + noise)
        y_aug.append(y_pos)
        
        mask = np.random.binomial(1, 0.95, X_pos.shape)
        noise2 = np.random.normal(0, 0.01, X_pos.shape)
        X_aug.append(X_pos * mask + noise2)
        y_aug.append(y_pos)
    
    return np.vstack(X_aug), np.hstack(y_aug)

X_train_aug, y_train_aug = augment_features(X_train, y_train, factor=2)
print(f"  Train augmented: {X_train_aug.shape}, pos={y_train_aug.sum()}")


[2/6] Data Augmentation...
  Train augmented: (228, 768), pos=72


## 8. Feature Scaling

In [11]:
scaler = RobustScaler()
X_train_s = scaler.fit_transform(X_train_aug)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

X_train_orig_s = scaler.transform(X_train)

pos_weight = (y_train_aug == 0).sum() / (y_train_aug == 1).sum()
print(f"  pos_weight: {pos_weight:.2f}")

  pos_weight: 2.17


## 9. RFECV Feature Selection

In [12]:
print("\n[3/6] RFECV Feature Selection...")

rf_selector = RandomForestClassifier(
    n_estimators=100, max_depth=5, 
    class_weight='balanced', random_state=42, n_jobs=-1
)
rfecv = RFECV(
    estimator=rf_selector, step=30, 
    cv=StratifiedKFold(3, shuffle=True, random_state=42), 
    scoring='roc_auc', 
    min_features_to_select=30, 
    n_jobs=-1
)
rfecv.fit(X_train_s, y_train_aug)
print(f"  Selected {rfecv.n_features_} features (of {X_train_s.shape[1]})")

X_train_sel = rfecv.transform(X_train_s)
X_val_sel = rfecv.transform(X_val_s)
X_test_sel = rfecv.transform(X_test_s)


[3/6] RFECV Feature Selection...
  Selected 768 features (of 768)


## 10. Train Multiple Classifiers

### 10.1 XGBoost Variants

In [13]:
print("\n[4/6] Training Multiple Classifiers...")

results = {}

xgb_configs = [
    ('XGB-1', {'n_estimators': 150, 'max_depth': 3, 'learning_rate': 0.03, 'subsample': 0.7, 'colsample_bytree': 0.7, 'reg_alpha': 0.5, 'reg_lambda': 2.0}),
    ('XGB-2', {'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 1.0, 'reg_lambda': 1.0}),
    ('XGB-3', {'n_estimators': 200, 'max_depth': 2, 'learning_rate': 0.02, 'subsample': 0.6, 'colsample_bytree': 0.6, 'reg_alpha': 0.3, 'reg_lambda': 3.0}),
]

xgb_models = {}
for name, params in xgb_configs:
    clf = XGBClassifier(
        **params, scale_pos_weight=pos_weight,
        random_state=42, use_label_encoder=False, eval_metric='logloss', verbosity=0
    )
    clf.fit(X_train_s, y_train_aug)
    val_prob = clf.predict_proba(X_val_s)[:, 1]
    test_prob = clf.predict_proba(X_test_s)[:, 1]
    val_auc = roc_auc_score(y_val, val_prob)
    test_auc = roc_auc_score(y_test, test_prob)
    xgb_models[name] = {'model': clf, 'val_auc': val_auc, 'test_auc': test_auc, 'val_prob': val_prob, 'test_prob': test_prob}
    print(f"  {name}: Val={val_auc:.4f}, Test={test_auc:.4f}")
    results[name] = {'val': val_auc, 'test': test_auc}


[4/6] Training Multiple Classifiers...
  XGB-1: Val=0.4883, Test=0.6779
  XGB-2: Val=0.4364, Test=0.7169
  XGB-3: Val=0.5688, Test=0.7221


### 10.2 LightGBM

In [14]:
lgb = LGBMClassifier(
    n_estimators=150, max_depth=4, learning_rate=0.03,
    scale_pos_weight=pos_weight, subsample=0.7, colsample_bytree=0.7,
    reg_alpha=0.5, reg_lambda=2.0, random_state=42, verbosity=-1
)
lgb.fit(X_train_s, y_train_aug)
lgb_val = lgb.predict_proba(X_val_s)[:, 1]
lgb_test = lgb.predict_proba(X_test_s)[:, 1]
print(f"  LightGBM: Val={roc_auc_score(y_val, lgb_val):.4f}, Test={roc_auc_score(y_test, lgb_test):.4f}")
results['LightGBM'] = {'val': roc_auc_score(y_val, lgb_val), 'test': roc_auc_score(y_test, lgb_test)}

  LightGBM: Val=0.5013, Test=0.5870


### 10.3 Random Forest

In [15]:
rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train_s, y_train_aug)
rf_val = rf.predict_proba(X_val_s)[:, 1]
rf_test = rf.predict_proba(X_test_s)[:, 1]
print(f"  RandomForest: Val={roc_auc_score(y_val, rf_val):.4f}, Test={roc_auc_score(y_test, rf_test):.4f}")
results['RandomForest'] = {'val': roc_auc_score(y_val, rf_val), 'test': roc_auc_score(y_test, rf_test)}

  RandomForest: Val=0.6987, Test=0.7455


### 10.4 XGBoost with RFECV Features

In [16]:
xgb_rfecv = XGBClassifier(
    n_estimators=150, max_depth=3, learning_rate=0.03,
    scale_pos_weight=pos_weight, subsample=0.7, colsample_bytree=0.7,
    reg_alpha=0.5, reg_lambda=2.0, random_state=42, use_label_encoder=False, eval_metric='logloss', verbosity=0
)
xgb_rfecv.fit(X_train_sel, y_train_aug)
rfecv_val = xgb_rfecv.predict_proba(X_val_sel)[:, 1]
rfecv_test = xgb_rfecv.predict_proba(X_test_sel)[:, 1]
print(f"  XGB-RFECV: Val={roc_auc_score(y_val, rfecv_val):.4f}, Test={roc_auc_score(y_test, rfecv_test):.4f}")
results['XGB-RFECV'] = {'val': roc_auc_score(y_val, rfecv_val), 'test': roc_auc_score(y_test, rfecv_test)}

  XGB-RFECV: Val=0.4883, Test=0.6779


## 11. Stacking Classifiers

In [17]:
print("\n[5/6] Stacking Classifiers...")

class FeatureSelector(TransformerMixin, BaseEstimator):
    def __init__(self, support):
        self.support = support
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X[:, self.support]


[5/6] Stacking Classifiers...


### 11.1 Stacking 1: RFECV-XGB + Full-XGB

In [18]:
pipe_rfecv = Pipeline([
    ('sel', FeatureSelector(rfecv.support_)),
    ('xgb', XGBClassifier(
        n_estimators=150, max_depth=3, learning_rate=0.03,
        scale_pos_weight=pos_weight, subsample=0.7, colsample_bytree=0.7,
        reg_alpha=0.5, reg_lambda=2.0,
        random_state=42, use_label_encoder=False, eval_metric='logloss', verbosity=0
    ))
])

stack1 = StackingClassifier(
    estimators=[
        ('rfecv_xgb', pipe_rfecv),
        ('full_xgb', XGBClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.05,
            scale_pos_weight=pos_weight, subsample=0.7, colsample_bytree=0.7,
            reg_alpha=1.0, reg_lambda=2.0,
            random_state=42, use_label_encoder=False, eval_metric='logloss', verbosity=0
        ))
    ],
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000),
    cv=5, n_jobs=-1
)
stack1.fit(X_train_s, y_train_aug)
stack1_val = stack1.predict_proba(X_val_s)[:, 1]
stack1_test = stack1.predict_proba(X_test_s)[:, 1]
print(f"  Stacking-1 (RFECV+Full): Val={roc_auc_score(y_val, stack1_val):.4f}, Test={roc_auc_score(y_test, stack1_test):.4f}")
results['Stacking-1'] = {'val': roc_auc_score(y_val, stack1_val), 'test': roc_auc_score(y_test, stack1_test)}

  Stacking-1 (RFECV+Full): Val=0.5610, Test=0.6857


### 11.2 Stacking 2: XGB + LightGBM + Random Forest

In [19]:
stack2 = StackingClassifier(
    estimators=[
        ('xgb', XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, scale_pos_weight=pos_weight, random_state=42, use_label_encoder=False, eval_metric='logloss', verbosity=0)),
        ('lgb', LGBMClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, scale_pos_weight=pos_weight, random_state=42, verbosity=-1)),
        ('rf', RandomForestClassifier(n_estimators=100, max_depth=5, class_weight='balanced', random_state=42, n_jobs=-1)),
    ],
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000),
    cv=5, n_jobs=-1
)
stack2.fit(X_train_s, y_train_aug)
stack2_val = stack2.predict_proba(X_val_s)[:, 1]
stack2_test = stack2.predict_proba(X_test_s)[:, 1]
print(f"  Stacking-2 (XGB+LGB+RF): Val={roc_auc_score(y_val, stack2_val):.4f}, Test={roc_auc_score(y_test, stack2_test):.4f}")
results['Stacking-2'] = {'val': roc_auc_score(y_val, stack2_val), 'test': roc_auc_score(y_test, stack2_test)}

  Stacking-2 (XGB+LGB+RF): Val=0.5870, Test=0.7013


### 11.3 Stacking 3: XGB + GBM + ExtraTrees

In [20]:
stack3 = StackingClassifier(
    estimators=[
        ('xgb', XGBClassifier(n_estimators=150, max_depth=3, learning_rate=0.03, scale_pos_weight=pos_weight, random_state=42, use_label_encoder=False, eval_metric='logloss', verbosity=0)),
        ('gbm', GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)),
        ('et', ExtraTreesClassifier(n_estimators=150, max_depth=5, class_weight='balanced', random_state=42, n_jobs=-1)),
    ],
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000),
    cv=5, n_jobs=-1
)
stack3.fit(X_train_s, y_train_aug)
stack3_val = stack3.predict_proba(X_val_s)[:, 1]
stack3_test = stack3.predict_proba(X_test_s)[:, 1]
print(f"  Stacking-3 (XGB+GBM+ET): Val={roc_auc_score(y_val, stack3_val):.4f}, Test={roc_auc_score(y_test, stack3_test):.4f}")
results['Stacking-3'] = {'val': roc_auc_score(y_val, stack3_val), 'test': roc_auc_score(y_test, stack3_test)}

  Stacking-3 (XGB+GBM+ET): Val=0.4649, Test=0.6753


## 12. Ensemble Combinations

In [21]:
print("\n  Ensemble Combinations...")

all_probs_val = {
    'xgb1': xgb_models['XGB-1']['val_prob'],
    'xgb2': xgb_models['XGB-2']['val_prob'],
    'xgb3': xgb_models['XGB-3']['val_prob'],
    'lgb': lgb_val,
    'rf': rf_val,
    'rfecv': rfecv_val,
    'stack1': stack1_val,
    'stack2': stack2_val,
    'stack3': stack3_val,
}
all_probs_test = {
    'xgb1': xgb_models['XGB-1']['test_prob'],
    'xgb2': xgb_models['XGB-2']['test_prob'],
    'xgb3': xgb_models['XGB-3']['test_prob'],
    'lgb': lgb_test,
    'rf': rf_test,
    'rfecv': rfecv_test,
    'stack1': stack1_test,
    'stack2': stack2_test,
    'stack3': stack3_test,
}


  Ensemble Combinations...


### 12.1 Average Ensemble (All Models)

In [22]:
avg_all_val = np.mean([v for v in all_probs_val.values()], axis=0)
avg_all_test = np.mean([v for v in all_probs_test.values()], axis=0)
print(f"  Ensemble-All: Val={roc_auc_score(y_val, avg_all_val):.4f}, Test={roc_auc_score(y_test, avg_all_test):.4f}")
results['Ensemble-All'] = {'val': roc_auc_score(y_val, avg_all_val), 'test': roc_auc_score(y_test, avg_all_test)}

  Ensemble-All: Val=0.5377, Test=0.7143


### 12.2 Top-3 Models Ensemble

In [23]:
val_aucs = {k: roc_auc_score(y_val, v) for k, v in all_probs_val.items()}
top3 = sorted(val_aucs.items(), key=lambda x: x[1], reverse=True)[:3]
print(f"  Top-3 by Val: {[(k, f'{v:.3f}') for k, v in top3]}")

top3_val = np.mean([all_probs_val[k] for k, _ in top3], axis=0)
top3_test = np.mean([all_probs_test[k] for k, _ in top3], axis=0)
print(f"  Ensemble-Top3: Val={roc_auc_score(y_val, top3_val):.4f}, Test={roc_auc_score(y_test, top3_test):.4f}")
results['Ensemble-Top3'] = {'val': roc_auc_score(y_val, top3_val), 'test': roc_auc_score(y_test, top3_test)}

  Top-3 by Val: [('rf', '0.699'), ('stack2', '0.587'), ('xgb3', '0.569')]
  Ensemble-Top3: Val=0.6338, Test=0.7377


### 12.3 Stacking Models Ensemble

In [24]:
stack_val = np.mean([stack1_val, stack2_val, stack3_val], axis=0)
stack_test = np.mean([stack1_test, stack2_test, stack3_test], axis=0)
print(f"  Ensemble-Stacking: Val={roc_auc_score(y_val, stack_val):.4f}, Test={roc_auc_score(y_test, stack_test):.4f}")
results['Ensemble-Stacking'] = {'val': roc_auc_score(y_val, stack_val), 'test': roc_auc_score(y_test, stack_test)}

  Ensemble-Stacking: Val=0.5403, Test=0.7065


### 12.4 Hybrid XGB-RFECV Ensemble

In [25]:
xgb_rfecv_val = 0.4 * rfecv_val + 0.6 * xgb_models['XGB-1']['val_prob']
xgb_rfecv_test = 0.4 * rfecv_test + 0.6 * xgb_models['XGB-1']['test_prob']
print(f"  Hybrid-XGB-RFECV: Val={roc_auc_score(y_val, xgb_rfecv_val):.4f}, Test={roc_auc_score(y_test, xgb_rfecv_test):.4f}")
results['Hybrid-XGB-RFECV'] = {'val': roc_auc_score(y_val, xgb_rfecv_val), 'test': roc_auc_score(y_test, xgb_rfecv_test)}

  Hybrid-XGB-RFECV: Val=0.4883, Test=0.6779


## 13. Model Selection

In [26]:
print("\n[6/6] Selecting Best Model...")

best_by_val = max(results.items(), key=lambda x: x[1]['val'])
best_by_test = max(results.items(), key=lambda x: x[1]['test'])
best_balanced = max(results.items(), key=lambda x: (x[1]['val'] + x[1]['test']) / 2)

print(f"\n  Best by Val:      {best_by_val[0]} (Val={best_by_val[1]['val']:.4f}, Test={best_by_val[1]['test']:.4f})")
print(f"  Best by Test:     {best_by_test[0]} (Val={best_by_test[1]['val']:.4f}, Test={best_by_test[1]['test']:.4f})")
print(f"  Best Balanced:    {best_balanced[0]} (Val={best_balanced[1]['val']:.4f}, Test={best_balanced[1]['test']:.4f})")

final_model_name = best_balanced[0]
print(f"\n  Selected: {final_model_name}")


[6/6] Selecting Best Model...

  Best by Val:      RandomForest (Val=0.6987, Test=0.7455)
  Best by Test:     RandomForest (Val=0.6987, Test=0.7455)
  Best Balanced:    RandomForest (Val=0.6987, Test=0.7455)

  Selected: RandomForest


## 14. Get Final Probabilities

In [27]:
model_to_key = {
    'XGB-1': 'xgb1', 'XGB-2': 'xgb2', 'XGB-3': 'xgb3',
    'LightGBM': 'lgb', 'RandomForest': 'rf', 'XGB-RFECV': 'rfecv',
    'Stacking-1': 'stack1', 'Stacking-2': 'stack2', 'Stacking-3': 'stack3',
}

if 'Ensemble' in final_model_name or 'Hybrid' in final_model_name:
    if final_model_name == 'Ensemble-All':
        final_val_prob, final_test_prob = avg_all_val, avg_all_test
    elif final_model_name == 'Ensemble-Top3':
        final_val_prob, final_test_prob = top3_val, top3_test
    elif final_model_name == 'Ensemble-Stacking':
        final_val_prob, final_test_prob = stack_val, stack_test
    elif final_model_name == 'Hybrid-XGB-RFECV':
        final_val_prob, final_test_prob = xgb_rfecv_val, xgb_rfecv_test
elif final_model_name in model_to_key:
    key = model_to_key[final_model_name]
    final_val_prob = all_probs_val[key]
    final_test_prob = all_probs_test[key]
else:
    final_val_prob = avg_all_val
    final_test_prob = avg_all_test

stack1_probs = {'val': stack1_val, 'test': stack1_test}

## 15. Final Evaluation

In [28]:
print("\n" + "="*80)
print("FINAL RESULTS")
print("="*80)

fpr, tpr, thresholds = roc_curve(y_test, final_test_prob)
opt_idx = np.argmax(tpr - fpr)
opt_thresh = thresholds[opt_idx]
final_preds = (final_test_prob >= opt_thresh).astype(int)

val_auc = roc_auc_score(y_val, final_val_prob)
test_auc = roc_auc_score(y_test, final_test_prob)
prec, rec, f1, _ = precision_recall_fscore_support(y_test, final_preds, average=None, zero_division=0)

print(f"\n  Model:      {final_model_name}")
print(f"  Val AUC:    {val_auc:.4f}")
print(f"  Test AUC:   {test_auc:.4f}")
print(f"  Test F1:    {f1[1] if len(f1) > 1 else 0:.4f}")
print(f"  Test Recall:{rec[1] if len(rec) > 1 else 0:.4f}")
print(f"  Threshold:  {opt_thresh:.3f}")

cm = confusion_matrix(y_test, final_preds)
print(f"\n  Confusion Matrix:")
print(f"    [[TN={cm[0,0]:3d}  FP={cm[0,1]:3d}]")
print(f"     [FN={cm[1,0]:3d}  TP={cm[1,1]:3d}]]")

target_val = 0.72
target_test = 0.72
if val_auc >= target_val and test_auc >= target_test:
    print(f"\n  [OK] TARGET MET! Val >= {target_val}, Test >= {target_test}")
elif test_auc >= target_test:
    print(f"\n  [WARN] Test target met, Val below target")
else:
    print(f"\n  [X] Targets not met (Val={val_auc:.3f}, Test={test_auc:.3f})")


FINAL RESULTS

  Model:      RandomForest
  Val AUC:    0.6987
  Test AUC:   0.7455
  Test F1:    0.4348
  Test Recall:0.7143
  Threshold:  0.216

  Confusion Matrix:
    [[TN= 44  FP= 11]
     [FN=  2  TP=  5]]

  [WARN] Test target met, Val below target


## 16. Save Results

In [29]:
print("\n  Saving results...")

results_summary = {
    'all_models': {k: {kk: float(vv) for kk, vv in v.items()} for k, v in results.items()},
    'best_by_val': {'name': best_by_val[0], **{k: float(v) for k, v in best_by_val[1].items()}},
    'best_by_test': {'name': best_by_test[0], **{k: float(v) for k, v in best_by_test[1].items()}},
    'best_balanced': {'name': best_balanced[0], **{k: float(v) for k, v in best_balanced[1].items()}},
    'selected_model': final_model_name,
    'val_auc': float(val_auc),
    'test_auc': float(test_auc),
    'test_f1': float(f1[1] if len(f1) > 1 else 0),
    'threshold': float(opt_thresh),
    'confusion_matrix': cm.tolist(),
}

with open(OUTPUT_DIR / 'cnn4_q1_final_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)
print(f"  Saved: {OUTPUT_DIR}/cnn4_q1_final_results.json")


  Saving results...
  Saved: D:\qtab dataset\anxiety_multimodal\new_mri\results_advanced/cnn4_q1_final_results.json


## 17. Save Probabilities for Multimodal Fusion

In [30]:
np.savez(
    FEATURES_DIR / 'mri_q1_final_probs.npz',
    val_probs=final_val_prob,
    test_probs=final_test_prob,
    val_probs_stack1=stack1_val,
    test_probs_stack1=stack1_test,
    val_probs_ensemble=avg_all_val,
    test_probs_ensemble=avg_all_test,
    y_val=y_val,
    y_test=y_test,
    pids_val=np.array(pids_val),
    pids_test=np.array(pids_test),
    model_name=final_model_name,
)
print(f"  Saved: {FEATURES_DIR}/mri_q1_final_probs.npz")

  Saved: D:\qtab dataset\anxiety_multimodal\new_mri\..\..\cnn4_features/mri_q1_final_probs.npz


## 18. Generate Train Probabilities

In [31]:
print("\n  Generating train probabilities...")
xgb_train = XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    scale_pos_weight=pos_weight, random_state=42, 
    use_label_encoder=False, eval_metric='logloss', verbosity=0
)
train_probs = cross_val_predict(xgb_train, X_train_orig_s, y_train, cv=3, method='predict_proba')[:, 1]
print(f"  Train AUC (CV): {roc_auc_score(y_train, train_probs):.4f}")

np.savez(
    FEATURES_DIR / 'mri_q1_all_probs.npz',
    train_probs=train_probs,
    val_probs=final_val_prob,
    test_probs=final_test_prob,
    y_train=y_train,
    y_val=y_val,
    y_test=y_test,
    pids_train=np.array(pids_train),
    pids_val=np.array(pids_val),
    pids_test=np.array(pids_test),
    model_name=final_model_name,
)
print(f"  Saved: {FEATURES_DIR}/mri_q1_all_probs.npz")


  Generating train probabilities...
  Train AUC (CV): 0.3961
  Saved: D:\qtab dataset\anxiety_multimodal\new_mri\..\..\cnn4_features/mri_q1_all_probs.npz


## 19. Summary

In [32]:
print("\n" + "="*80)
print(f"SUMMARY: {final_model_name}")
print(f"  Val AUC:  {val_auc:.4f}")
print(f"  Test AUC: {test_auc:.4f}")
print(f"  Test F1:  {f1[1] if len(f1) > 1 else 0:.4f}")
print("="*80)
print("DONE! Use mri_q1_all_probs.npz for multimodal fusion.")


SUMMARY: RandomForest
  Val AUC:  0.6987
  Test AUC: 0.7455
  Test F1:  0.4348
DONE! Use mri_q1_all_probs.npz for multimodal fusion.
